# Algorytmika i matematyka uczenia maszynowego 
## Laboratorium 11

### Zadanie 1

Zaimplementuj systemu rekomendacji filmów w oparciu o indeks Jaccarda. System ma za zadanie zwrócić listę filmów sugerowany dla podanego użytownika.

Dane zostały pobrane z serwisu Kaggle z https://www.kaggle.com/datasets/gargmanas/movierecommenderdataset

Zbiór zawiera dwa pliki:
- `movies.csv` lista filmów wraz z ich identyfikatorami
- `ratings.csv` lista ocen filmów przez użytkowników

**Zadanie:**
* Wczytaj oba pliki.
* Zamień wszystkie oceny użytkownika na wartość 1 (zastosuj próg okreśjący czy film się podobał czy nie np. 3).
* Stwórz macierz ocen użytkowników w której wierszach będą użytkownicy, a w kolumnach filmy. Wartość w macierzy jest flagą mówiącą czy użytkownikowi film się podobał czy nie. 
* Wypełnij brakujące wartości zerami.
* Utwórz macierz podobieństwa Jaccarda pomiędzy użytkownikami (każdy z każdym).
    - Możesz wykorzystać funkcję [jaccard](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.jaccard.html) z biblioteki scipy.
* Zaimplementuj funkcję która dla podanego użytkownika zwróci listę sugerowanych filmów.
    - Funkcja powinna zwrócić listę filmów które nie były ocenione przez użytkownika, a które są rekomendowane dla niego.



In [9]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import distance

In [10]:
movies = pd.read_csv('./movies.csv')
ratings = pd.read_csv('./ratings.csv')

In [11]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [12]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [13]:
ratings_bool = ratings.copy()
ratings_bool['rating'] = np.where(ratings['rating'] > 3, 1, 0)
ratings_bool['rating'].value_counts()

rating
1    61716
0    39120
Name: count, dtype: int64

In [14]:
user_movie = ratings_bool.pivot_table(index='userId', columns='movieId', values='rating', fill_value=0)
user_movie

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
607,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
608,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
user_ids = user_movie.index.tolist()
n_users = len(user_ids)
jaccard_matrix = np.zeros((n_users, n_users))
jaccard_matrix.shape

(610, 610)

In [16]:
for i in range(n_users):
    for j in range(n_users):
        if i == j:
            jaccard_matrix[i, j] = 1.0
        else:
            jaccard_matrix[i, j] = 1.0 - distance.jaccard(user_movie.iloc[i], user_movie.iloc[j])

In [17]:
len(jaccard_matrix[0])

610

In [23]:
def recommend_movies(user_id, top_n=5):
    idx = user_ids.index(user_id)
    
    similar = jaccard_matrix[idx]
    similar[idx] = 0 # skip self-comparison
    
    similar_users_idx = np.argsort(similar)[::-1]
    already_rated_movies = set(ratings[ratings['userId'] == user_id]['movieId'].values)
    
    recommends = []
    for other_idx in similar_users_idx:
        already_rated_movies_other = set(ratings[ratings['userId'] == other_idx]['movieId'].values)
        
        for movie in already_rated_movies_other:
            if movie not in already_rated_movies and movie not in recommends:
                recommends.append(movie)
                if len(recommends) >= top_n:
                    break 
        if len(recommends) >= top_n:
                    break 
    
    return movies[movies['movieId'].isin(recommends)]['title']
    
recommend_movies(user_id = 2, top_n = 5)

5917     Batman Begins (2005)
7043     Hangover, The (2009)
8377    3 Days to Kill (2014)
8431           Blended (2014)
9280           Snowden (2016)
Name: title, dtype: object



### Zadanie 2


Algorytm MinHash na przykładzie wykrywania plagiatów

Wykonaj kolejno następujące kroki:

1. Pobierz 8 akapitów tekstu (nie za krótkich), każdy o różnej tematyce (mogą być np. z różnych haseł Wikipedii), trzymaj się jednego języka (np. PL lub ENG). Wklej je do jednego pliku tekstowego, z linią wolną jako separatorem.

2. Skopiuj wybrane 2-3 akapity i ręcznie nieco zmodyfikuj.

> Przykład (z hasła https://pl.wikipedia.org/wiki/Fryderyk_Chopin):

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych polskich kompozytorów w historii. Był jednym z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu. Elementem charakterystycznym dla utworów Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców stylistycznych polskiej muzyki ludowej.```

↓↓↓ Zmieniono na ↓↓↓

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych kompozytorów polskich w historii.  Był jednym  z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu! Elementem charakterystycznym dla utworów Fryderyka Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców polskiej muzyki ludowej.```

Otrzymasz zatem w pliku tekstowym 10 lub 11 akapitów tekstu (kolejność dowolna, te „splagiatowane” nie muszą być na końcu).

3. Z poziomu skryptu: wczytaj wszystkie akapity z pliku. Zbuduj 100 "losowych" funkcji haszujących.

> Sugestia: funkcją "bazową" jest po prostu `hash(...)`. Zakładamy 64-bitową wersję Pythona 3.x, wtedy `hash(...)` jest 64-bitowy.

Na liście seeds umieszczamy 100 losowych liczb 64-bitowych. Aby obliczyć $i$-ty hash dla ciągu 
należy wykonać `hash(s) ^ seeds[i]` (użycie operatora XOR).

4. Przyjmij niewielką wartość $Q$ (np. 15) i dla każdego akapitu
    - oznacz jego długość przez $n$,
    - dla każdej ze 100 funkcji haszujących policz hasza w przesuwnym oknie tekstu o długości $Q$ znaków (czyli łącznie mamy $n - Q + 1$ wartości hasza); zapamiętaj MINIMUM z tych $n - Q + 1$
 wartości.
Na wyjściu mamy zatem (dla 11 akapitów) 11 * 100 wartości haszy.

5. Rozważ pary akapitów "każdy z każdym". Jeśli dla danej pary co najmniej (np.) 30 haszy jest wspólnych, to uważamy akapity za podobne (być może plagiat) i wyświetlamy na ekranie.

6. Wyświetl czas obliczeń (powinien wynosić mniej niż 0.5s).

7. Poeksperymentuj z liczbą użytych funkcji haszujących, wartością, stopniem modyfikacji oryginalnych akapitów tekstu, progiem detekcji akapitów podobnych.

In [2]:
# load the paragraphs from the file
def load_paragraphs(file_path):
    with open(file_path, 'r') as f:
        paragraphs = f.read().split('\n\n')
    return paragraphs

paragraphs = load_paragraphs('random_articles.txt')

In [3]:
len(paragraphs)

11

In [7]:
# 100 hash functions
num_hashes = 100
threshold = 30
n_paragraphs = len(paragraphs)
Q = 15 # window size
np.random.seed(420)
seeds = np.random.randint(2**62, 2**63, size=num_hashes, dtype=np.int64)

def minhash(paragraph, seeds, Q):
    n = len(paragraph)
    res = []
    for seed in seeds:
        hashes = []
        for i in range(n - Q + 1):
            window = paragraph[i:i + Q]
            h = hash(window) ^ seed
            hashes.append(h)
        res.append(min(hashes))
    return res

start = time.time()    
minhash_all = [minhash(p, seeds, Q) for p in paragraphs]
for i in range(n_paragraphs):
    for j in range(i + 1, n_paragraphs):
        common = np.sum(np.array(minhash_all[i]) == np.array(minhash_all[j]))
        if common >= threshold:
            print(f"Similar paragraphs: {i}, {j} - hash value: {common}.")
end = time.time()
print(f"Time: {end - start:.2f} s")

Similar paragraphs: 0, 3 - hash value: 53.
Similar paragraphs: 1, 6 - hash value: 34.
Similar paragraphs: 5, 9 - hash value: 46.
Time: 0.14 s
